In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv
import urllib.parse
import os
import clickhouse_connect
import numpy as np
import pandas as pd
import datetime as dt

orange = '#FF670F'
pink = '#FC006D'
blue = '#006CED'
green = '#2BD67C'
red = '#FF5D5D'
black = '#2B3034'

#tables
grdf_all='datamart_grdf.dataset_grdf_all'
releves='datawarehouse_prod.fact_payload'
clients='datawarehouse_prod.dim_entity'
equipements='datawarehouse_prod.dim_device'
capteurs='datawarehouse_prod.dim_sensor'
patrimoine='datawarehouse_prod.device_patrimonial_data'

source="grdf"
maxi = 10000000000000

load_dotenv()
host = urllib.parse.quote_plus(os.environ.get("clk_prod"))
username = urllib.parse.quote_plus(os.environ.get("USER_NAME"))
password = urllib.parse.quote_plus(os.environ.get('clk_prod_pass'))
client = clickhouse_connect.get_client(host=host, username=username, 
                                       port = 443, password=password)

<span style="color:red">

### Table of Contents

* [1 - Table datamart_grdf](#grdf)
    * [1.1 Différence entre DB grdf et payload grdf](#11)
    * [1.2 Volume de PCE et relevés par statut d'équipement](#12)
    * [1.3 Volume de données publiée en fonction des fréquences](#13)
    * [1.4 Vérification des noms des PCE parmis les PCE en fonctionnement](#14)
* [2 - Exploration des données de la table payload](#payloads)
    * [2.1 Liste des capteurs](#21)
    * [2.2 Distribution globale des relevés](#22)
    * [2.3 Exploration des valeurs de consommation de gaz (Relevés parus depuis 01/2024)](#23)
    * [2.4 Analyse des types de valeurs des relevés, tous capteurs confondus](#24)
    * [2.5 Description des fréquences des relevés pour les capteurs liés aux équipements de la famille 'Compteur Gaz'](#25)
    * [2.6 Couverture des relevés (famille 'Compteur Gaz' données numériques)](#26)
    * [2.7 Délai d'ingestion des relevés](#27)
    * [2.8 Types de dates](#28)
    * [2.9 Données chevauchantes](#29)
* [3 - Recherche d'anomalies parmi les données publiées](#anomalies1)
* [4 - Recherche d'anomalies parmi les données informatives](#anomalies2) 
</span>

# <span style="color:purple"> 1 - Exploration données de la table datamart_grdf </span> 

## <span style="color:green"> 1 - Différence entre DB grdf et payload grdf </span> 

In [ ]:
df = client.query_df(f"""SELECT iot_entity, count(time) 
        FROM {grdf_all} 
        GROUP BY iot_entity
        settings max_rows_to_read={maxi};""")

In [ ]:
print(f"La table {grdf_all} compte {df.shape[0]} clients et {df['count(time)'].sum()} relevés.")
print("Pour comparaison la table payload dont les relevés appartenaient à la source 'grdf' contient 236 clients et plus de 1 million de relevés.")

## <span style="color:green"> 2 - Volume de PCE et relevés par statut d'équipement</span> 

In [ ]:
df_status = client.query_df(f"""SELECT device_status, count(distinct(device_id)) nbPCE, count(distinct time) nbReleves
FROM {grdf_all}
GROUP BY device_status
settings max_rows_to_read={maxi};""")

In [ ]:
df_status

## <span style="color:green"> 3 - Volume de données publiée en fonction des fréquences</span> 

In [ ]:
df = client.query_df(f"""
    SELECT frequency, grdf_frequency, count(distinct device_id) nbdev, count(distinct iot_entity) nbent, count(distinct time) nbpay
    FROM {grdf_all}
    GROUP BY frequency, grdf_frequency
    SETTINGS max_rows_to_read = {maxi};""")
df

In [ ]:
df_sans_freq

In [ ]:
df_sans_freq[['device_id', 'id_PCE']].to_csv('./data/grdf_PCE_sans_frequence.csv', sep=';', index=False)

In [ ]:
df_sans_freq = client.query_df(f"""
    SELECT DISTINCT g.device_id, g.frequency, g.grdf_frequency, arrayElement(d.reference_value, 1) id_PCE
    FROM {grdf_all} g
    LEFT JOIN {equipements} d
    ON g.device_id = d.device_id
    WHERE frequency = ''
    SETTINGS max_rows_to_read = {maxi};""")

### <span style="color:blue"> 2 - a) Vérification des fréquences 1M (parmi les PCE en fonctionnement)</span>

In [ ]:
df_1M = client.query_df(f"""SELECT p.device_id, p.device_name, arrayReduce('avg', diffs) AS avgDiff 
    FROM(
        SELECT 
            p.device_id,
            p.device_name,
            arrayDifference(arraySort(groupArray(p.time))) AS diffs
        FROM {grdf_all} p
        WHERE frequency = 'M' AND device_status = 'working'
        GROUP BY
        p.device_id, 
        p.device_name) p
    ORDER BY avgDiff DESC
    settings max_rows_to_read = {maxi}""")

In [ ]:
df_1M['timedelta'] = df_1M.avgDiff.apply(lambda x: dt.timedelta(seconds=x))
max_val = df_1M.head(100).avgDiff.max()
tickvals = [-float('inf'), 1, 60, 
            60*5, 60*10, 60*30, 60*45,
            3600, 3600*6, 3600*12, 
            86400, 86400*5, 86400*15, 86400*30, max_val]
ticktext = ['< 1s', '< 1mn', '(1mn - 5mn])', '(5mn - 10mn])', '(10mn - 30mn])', 
           '(30mn - 45mn])', '(45mn - 1h])', '(1h - 6h])', '(6h - 12h])', 
           '(12h - 1jour])', '(1jour - 5jours])', '(5jours - 15jours])', 
           '(15jours - 30jours])', '> 1mois']
df_1M["interval"] = pd.cut(df_1M["avgDiff"], bins=tickvals, labels=ticktext)
counts = df_1M["interval"].value_counts(sort=False)

In [ ]:
df_1M[df_1M.interval=='(1jour - 5jours])']

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': pink}
    )
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Fréquence moyenne des relevés par couples équipement-capteur",
                  xaxis_title="fréquence moyenne", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
df_6M = client.query_df(f"""SELECT p.device_id, p.device_name, arrayReduce('avg', diffs) AS avgDiff 
    FROM(
        SELECT 
            p.device_id,
            p.device_name,
            arrayDifference(arraySort(groupArray(p.time))) AS diffs
        FROM {grdf_all} p
        WHERE frequency = '6M' AND device_status = 'working'
        GROUP BY
        p.device_name,
        p.device_id) p
    ORDER BY avgDiff DESC
    settings max_rows_to_read = {maxi}""")

In [ ]:
df_6M['timedelta'] = df_1M.avgDiff.apply(lambda x: dt.timedelta(seconds=x))
max_val = df_1M.head(100).avgDiff.max()
tickvals = [-float('inf'), 1, 60, 
            60*5, 60*10, 60*30, 60*45,
            3600, 3600*6, 3600*12, 
            86400, 86400*5, 86400*15, 86400*30, max_val]
ticktext = ['< 1s', '< 1mn', '(1mn - 5mn])', '(5mn - 10mn])', '(10mn - 30mn])', 
           '(30mn - 45mn])', '(45mn - 1h])', '(1h - 6h])', '(6h - 12h])', 
           '(12h - 1jour])', '(1jour - 5jours])', '(5jours - 15jours])', 
           '(15jours - 30jours])', '> 1mois']
df_1M["interval"] = pd.cut(df_1M["avgDiff"], bins=tickvals, labels=ticktext)
counts = df_1M["interval"].value_counts(sort=False)

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': pink}
    )
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Fréquence moyenne des relevés par couples équipement-capteur",
                  xaxis_title="fréquence moyenne", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### <span style="color:blue"> 2 - a) Vérification des fréquences 1M (parmi les PCE en fonctionnement)</span> 

## <span style="color:green"> 5 - Vérification des noms des PCE (parmi les PCE en fonctionnement)</span> 

In [ ]:
PCE = client.query_df(f"""
    SELECT entity_name, device_name, device_id, count(distinct time) nbReleves, min(time) debut, max(time) fin
    FROM {grdf_all}
    WHERE device_status IN ('working', 
    AND NOT match(device_name, '.*(?:[0-9]{{14}}.*|_?GI[0-9]{{6}}).*')
    GROUP BY entity_name, device_name, device_id
    settings max_rows_to_read={maxi};""")
PCE

130 PCE n'ont pas été identifié selon les règles indiquées par grdf.
Nombre de relevés par client dans ce cas de figure:

In [ ]:
PCE.groupby('entity_name').nbReleves.sum()

In [ ]:
list_PCE = PCE.device_id.to_list()
PCE_device = client.query_df(f"""
SELECT device_id, reference_value
FROM {equipements}
WHERE device_id IN {list_PCE}
settings max_rows_to_read={maxi};""")
PCE_device

Visiblement la reference value retrouvée pour ces équipements PCE dans dim_device correspond bien à la convention de nommage imposée par GRDF.

# <span style="color:purple"> 2 - Exploration des données de la table payload </span> 

## <span style="color:green"> 1 - Liste des capteurs </span> 

In [ ]:
sens = client.query_df(f"""SELECT DISTINCT p.payload_metric_id 
FROM {releves} p 
WHERE payload_source = 'grdf'
SETTINGS max_rows_to_read = {maxi};""")

In [ ]:
# no_name = sens.sort_values('payload_metric_id')[:17].payload_metric_id.to_list()
with_name = sens.sort_values('payload_metric_id')[17:].payload_metric_id.to_list()

In [ ]:
df = client.query_df(f"""SELECT s.name, p.payload_metric_id, min(p.payload_timestamp) debut, max(p.payload_timestamp) fin, count() nb
FROM {releves} p
LEFT JOIN {capteurs} s
ON p.payload_metric_id = s.sensor_id
WHERE p.payload_source = '{source}'
GROUP BY p.payload_metric_id, s.name
SETTINGS max_rows_to_read = {maxi};""")

In [ ]:
df = client.query_df(f"""SELECT s.name, p.payload_metric_id, min(p.payload_timestamp) debut, max(p.payload_timestamp) fin, count() nb
FROM {releves} p
LEFT JOIN {capteurs} s
ON p.payload_metric_id = s.sensor_id
WHERE p.payload_source = '{source}'
GROUP BY p.payload_metric_id, s.name
SETTINGS max_rows_to_read = {maxi};""")

In [ ]:
df.sort_values('nb')

## <span style="color:green"> 2 - Distribution globale des relevés </span> 

### <span style="color:blue"> a)- Distribution  des données par client </span> 

<span style="color:red"> **En se servant de l'incateur de source fournie par les relevés** </span> 

In [ ]:
df = client.query_df(f"""SELECT p.entity_id, p.device_id, p.payload_metric_id, count(p.payload_timestamp) nbPay\
                            FROM {releves} p
                            WHERE p.payload_source = '{source}'
                            GROUP BY p.entity_id, p.device_id, p.payload_metric_id\
                            ORDER BY nbPay
                            settings max_rows_to_read = {maxi}""")

In [ ]:
cnt = len(df.entity_id.unique())
avg = df.groupby('entity_id').nbPay.sum().mean().astype(int)
avg2 = df.groupby('entity_id').nbPay.count().mean().astype(int)
avg3 = df.groupby(['entity_id', 'device_id'], group_keys=False).count().groupby('entity_id').nbPay.count().mean().astype(int)
avg4 = df.groupby(['entity_id', 'payload_metric_id'], group_keys=False).count().groupby('entity_id').nbPay.count().mean().astype(int)
print(f"Nombre total de clients: {cnt}.")
print(f"Moyenne du nombre de relevés par client: {avg}.")
print(f"Moyenne du nombre de couples 'equipement-capteur' par client: {avg2}.")
print(f"Moyenne du nombre d'équipements par client: {avg3}.")
print(f"Moyenne du nombre de capteurs par client {avg4}.")

<span style="color:red"> **En se servant de l'indicateur de source fourni par la table des équipements** </span> 

In [ ]:
df_pay = df
df = client.query_df(f"""SELECT p.entity_id, p.device_id, p.payload_metric_id, count(p.payload_timestamp) nbPay
                            FROM {releves} p
                            INNER JOIN (SELECT device_id, reference_source
                                    FROM {equipements}
                                    WHERE has(reference_source, '{source}')) d
                            on d.device_id = p.device_id
                            GROUP BY p.entity_id, p.device_id, p.payload_metric_id                           
                            ORDER BY nbPay
                            settings max_rows_to_read = {maxi}""")

In [ ]:
cnt = len(df.entity_id.unique())
avg = df.groupby('entity_id').nbPay.sum().mean().astype(int)
avg2 = df.groupby('entity_id').nbPay.count().mean().astype(int)
avg3 = df.groupby(['entity_id', 'device_id'], group_keys=False).count().groupby('entity_id').nbPay.count().mean().astype(int)
avg4 = df.groupby(['entity_id', 'payload_metric_id'], group_keys=False).count().groupby('entity_id').nbPay.count().mean().astype(int)
print(f"Nombre total de clients: {cnt}.")
print(f"Moyenne du nombre de relevés par client: {avg}.")
print(f"Moyenne du nombre de couples 'equipement-capteur' par client: {avg2}.")
print(f"Moyenne du nombre d'équipements par client: {avg3}.")
print(f"Moyenne du nombre de capteurs par client {avg4}.")

In [ ]:
pas_ds_dev = set(df_pay.device_id) - set(df.device_id)
pas_ds_pay = set(df.device_id) - set(df_pay.device_id)
print(f"{len(pas_ds_dev)} équipements dont la source 'grdf' est indiquée dans les relevés qu'ils renvoient mais pas dans la table des équipements.")
print(f"{len(pas_ds_pay)} équipements dont la source 'grdf' est indiquée dans la table des équipements mais pas dans les relevés qu'ils renvoient.")

In [ ]:
df = client.query_df(f""" 
    SELECT device_id, device_type, reference_source
    FROM datawarehouse_prod.dim_device
    WHERE device_id IN {pas_ds_dev}
    SETTINGS max_rows_to_read = {maxi};

In [ ]:
avg = df.groupby('entity_id').nbPay.sum()
avg2 = df.groupby('entity_id').nbPay.count()
avg3 = df.groupby(['entity_id', 'device_id'], group_keys=False).count().groupby('entity_id').nbPay.count()
avg4 = df.groupby(['entity_id', 'payload_metric_id'], group_keys=False).count().groupby('entity_id').nbPay.count()
fig = go.Figure()
fig.add_trace(go.Box(name="Nombre de relevés moyen.", y=avg, marker={'color': blue}))
fig.add_trace(go.Box(name="Nombre de couple 'equip-capt'.", y=avg2, marker={'color': blue}))
fig.add_trace(go.Box(name="Nombre d'équipements moyen.", y=avg3, marker={'color': blue}))
fig.add_trace(go.Box(name = "Nombre de capteurs moyen.", y=avg4, marker={'color': blue}))
fig.update_yaxes(type="log")
fig.update_layout(title=f"Distribution des relevés / équipements / capteurs par client.",
              xaxis_title="Type de mesure par client", 
              yaxis_title="Nombre moyen <br> (Echelle logarithmique)",
              font_family="Arial", font_color=black, title_font_family="Times New Roman", 
                title_font_color=black, width=1000, height=400,
                template="simple_white", showlegend=False)
fig.show()

### <span style="color:blue"> b)- Distribution par combinaison équipement - capteur </span> 

In [ ]:
df = client.query_df(f"""SELECT p.device_id, p.payload_metric_id, count(p.payload_timestamp) nbPay
                            FROM {releves} p
                            WHERE p.payload_source = '{source}'
                            GROUP BY p.device_id, p.payload_metric_id
                            ORDER BY nbPay
                            settings max_rows_to_read = {maxi}""")

In [ ]:
print(f"Nombre total de couples 'equipement-capteur': {df.shape[0]}. En moyenne {df.nbPay.mean().astype(int)} relevés par couple.")
cnt = len(df.device_id.unique())
avg = df.groupby('device_id').nbPay.sum().mean().astype(int)
avg2 = df.groupby('device_id').nbPay.count().mean().astype(int)
print(f"Nombre total d'équipements {cnt}, qui en moyenne ont {avg} relevés et sont reliés à {avg2} capteurs.")
cnt = len(df.payload_metric_id.unique())
avg = df.groupby('payload_metric_id').nbPay.sum().mean().astype(int)
avg2 = df.groupby('payload_metric_id').nbPay.count().mean().astype(int)
print(f"Nombre de capteurs: {cnt}, qui en moyenne ont {avg} relevés et sont reliés {avg2} equipements.")

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Histogram(nbinsx = 200, x = df.nbPay.sort_values(),
        marker={'color': blue}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Distribution des relevés en fonction des paires 'équipement - capteur'",
                  xaxis_title="Nombre de relevés", 
                  yaxis_title="Nombre de paires 'équipement - capteur' <br> (Echelle logarithmique)",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### <span style="color:blue"> c)- Distribution par famille d'équipement </span> 

In [ ]:
df_familles = client.query_df(f"""
SELECT d.device_type, count(p.payload_timestamp) nbPay
FROM {releves} p
INNER JOIN (SELECT device_type, device_id
		FROM {equipements}) d	
ON p.device_id = d.device_id
WHERE p.payload_source = '{source}'
GROUP BY d.device_type
settings max_rows_to_read = {maxi};""")

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x = df_familles.device_type, y=df_familles.nbPay, text = df_familles.nbPay,
        marker={'color': blue}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Distribution des relevés GRDF en fonction de la famille d'équipement",
                  xaxis_title="Famille d'équipement", 
                  yaxis_title="Nombre de relevés",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### <span style="color:blue"> d)- Distribution par type de bâtiment </span> 

In [ ]:
df_build = client.query_df(f"""
        SELECT b.site_typology,
                b.sub_nature, 
                count(distinct p.entity_id) ent, 
                count(distinct p.device_id) dev, 
                count(distinct p.payload_metric_id) sens, 
                count(distinct p.payload_timestamp) nbPay
        FROM {releves} p
        INNER JOIN (SELECT sub_nature, site_typology, device_id
        		FROM {patrimoine}) b
        ON b.device_id = p.device_id
        WHERE p.payload_source = '{source}'
        GROUP BY b.site_typology, b.sub_nature
        settings max_rows_to_read = {maxi};""")
df_build

In [ ]:
print(f"Liste des différentes typologies de site rencontré dans les bâtiments ayant des relevés grdf:")
df_build.site_typology.drop_duplicates().to_list()

<span style="color:orange"> Il y existe un certain nombre d'équipements dont la typologie du site ou la sous nature n'est pas renseignée. Dans le pire des cas, il existe **153 clients** et **6700 équipements** qui n'ont aucun de ces deux champs renseignés </span>

In [ ]:
df_build[(df_build.site_typology.isna())|(df_build.site_typology=="")].sort_values('ent').tail(10)

<span style="color:orange"> On se concentre uniquement sur les typologies de sites suivants :</span> <br> 
* <span style="color:orange">'Equipements culturels',</span>
* <span style="color:orange">'Tertiaire',</span>
* <span style="color:orange">'Résidentiel',</span>
* <span style="color:orange">'Industriel',</span>
* <span style="color:orange">'Salles de quartier',</span>
* <span style="color:orange">'Batiments communaux (service mairie)',</span>
* <span style="color:orange">'Scolaires',</span>
* <span style="color:orange">'Mixte Tertiaire Industriel',</span>
* <span style="color:orange">'Logement',</span>
* <span style="color:orange">'Equipements sportifs',</span>
* <span style="color:orange">'Batiment CCAS',</span>
* <span style="color:orange">'Tertiaire ',</span>
* <span style="color:orange">'Gestion immobilière', </span>
* <span style="color:orange">'Crèche'] </span>

In [ ]:
list_typo = ['Equipements culturels','Tertiaire','Résidentiel','Industriel','Salles de quartier','Batiments communaux (service mairie)','Scolaires',
 'Mixte Tertiaire Industriel','Logement','Equipements sportifs','Batiment CCAS','Tertiaire ','Gestion immobilière', 'Crèche']

In [ ]:
df_build[(df_build.site_typology.isin(list_typo))].groupby('site_typology')[['ent', 'sens', 'dev', 'nbPay']].sum().sort_values('ent', ascending=False)

### <span style="color:blue"> e)- Recherche des doublons par famille d'équipement </span>

In [ ]:
dupliq_df = []
for fam in df_familles.device_type.to_list():
    doublons = client.query_df(f"""
        SELECT
            device_id,
            payload_metric_id,
            countIf(c > 1) total_doublons
        FROM (  SELECT
                    device_id,
                    payload_metric_id,
                    payload_timestamp,
                    count() AS c
                FROM {releves} p
                INNER JOIN {equipements} d 
                    ON p.device_id = d.device_id
                WHERE d.device_type = '{fam}' AND p.payload_source = '{source}'
                GROUP BY device_id, payload_metric_id, payload_timestamp
                )
        GROUP BY device_id, payload_metric_id
        ORDER BY total_doublons DESC
        settings max_rows_to_read = {maxi};""")
    nb = doublons[doublons.total_doublons>0].shape
    avg = doublons[doublons.total_doublons>0].total_doublons.mean()
    dupliq_df.append({"device_type":fam, "nb_couples":nb, "avg_nb_doublons":avg})

<span style="color:orange"> Aucun doublon quelle que soit la famille d'équipement </span>

### <span style="color:blue"> f)- Recherche des **équipements inactifs** (pas de relevés dans la collection payload) </span> 

In [ ]:
cnt_dev = client.query_df(f"""
    SELECT DISTINCT d.device_id, d.device_type
    FROM {equipements} d
    WHERE d.device_type IN {list_fam}
    SETTINGS max_rows_to_read = {maxi}""")
df_null = client.query_df(f"""
    SELECT d.device_id, d.device_type, f.device_id pay
    FROM {equipements} d
    LEFT JOIN(
    	SELECT DISTINCT device_id
    	FROM {releves}) f
    ON d.device_id = f.device_id
    WHERE d.device_type IN {list_fam} AND pay IS NULL
    SETTINGS join_use_nulls = 1, max_rows_to_read = {maxi}""")
print(f"{df_null.shape[0]} équipements dans les familles {list_fam} n'ont pas de relevé dans la table payload")
print(f"Cela représente {int(df_null.shape[0]/cnt_dev.shape[0]*100)} % des équipements appartenant à cette liste de familles")

**Nombre d'équipements par famille n'ayant aucun relevé dans la table payload**

In [ ]:
pd.DataFrame(df_null.groupby('device_type').device_id.count().sort_values(ascending=False))

## <span style="color:green"> 3 - Exploration des valeurs de consommation de gaz  <br> (Relevés parus depuis 01/2024)</span> 

In [ ]:
# Capteurs de consommation
V_reel_info = '64f9dc4e964b1c6b3092aebe'
V_reel_pub = '62447752f340cc25cbd2afb0'
V_brut_info = '64f9dbc06bd36472e445b9c7'
V_brut_pub = '62447746f340cc25cbd2afaf'

idx_reel_info = '64f9dad0bfcf77e14e7ab3bf'
idx_reel_pub = '62447396f340cc25cbd2afa2'
idx_brut_info = 'INFORM_RAW_INDEX_VALUE'
idx_brut_pub = '62447388f340cc25cbd2afa1'

date_fin_pub = 'CONSUMPTION_END_DATE'
date_debut_pub = 'CONSUMPTION_START_DATE'

date_fin_info = 'INFORM_CONSUMPTION_END_DATE'
date_debut_info = 'INFORM_CONSUMPTION_START_DATE'

In [ ]:
def detect_outliers_iqr(data, k = 2.5, return_thresholds=False):
    '''
    Detect outliers in a dataset using the interquartile range (IQR) method.
    '''
    # Calculate quartiles
    q25, q75 = np.percentile(data, [25, 75])
    # Calculate the IQR
    iqr = q75 - q25
    # Calculate the outlier cutoff
    cutoff = iqr * k
    # Calculate the lower and upper bounds
    thres = q75 + cutoff

    if return_thresholds:
        return lower_bound, upper_bound
    else:
        # Identify outliers
        outliers = data > upper_bound
        return outliers

In [ ]:
dev_freq = client.query_df(f"""
        SELECT DISTINCT device_id, frequency
        FROM {grdf_all}
        settings max_rows_to_read = {maxi}""")

### <span style="color:blue"> a)- Point de départ sur capteur 'Volume brut publié'</span>

In [ ]:
# frequence des équipements
couv_v_brut = client.query_df(f"""
        SELECT p.device_id, first_value(g.frequency) frequence, g.numero_pdl, min(p.payload_timestamp) debut, max(p.payload_timestamp) fin
        FROM {releves} p
            LEFT JOIN {grdf_all} g
            ON p.device_id = g.device_id
        WHERE p.payload_metric_id = '{V_brut_pub}' 
        GROUP BY p.device_id, g.numero_pdl
        settings max_rows_to_read = {maxi}""")

date_range = pd.date_range(start=couv_v_brut.debut.min(), end=couv_v_brut.fin.max(), freq="MS")
time_coverage_df = []
for ma_date in date_range:
    nbDev = couv_v_brut[(couv_v_brut.debut < ma_date) &  (couv_v_brut.fin > ma_date)].shape[0]
    time_coverage_df.append({
        'date':ma_date, "nbPCE":nbDev})
time_coverage_df = pd.DataFrame(time_coverage_df)

fig = go.Figure(go.Histogram(histfunc='avg', 
                             x=time_coverage_df.date, y=time_coverage_df.nbPCE, 
                            marker={'color': green}))
# Set xaxis range
fig.update_xaxes(type='date', 
                 tickformat='%Y-%m-%d', 
                 nticks=10, 
                 range=[couv_v_brut.debut.min(), couv_v_brut.fin.max()])
fig.update_yaxes(type='log')
fig.update_layout(title=f"Distribution des PCE relevant des données de volume brut publiés sur le temps",
                  xaxis_title="Date", 
                  yaxis_title=" # PCE associés à des volumes bruts publiés <br> (Echelle logarithmique)",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

<span style="color:orange">
Dans la route 'volume brut publié' (62447746f340cc25cbd2afaf), il existe des valeurs non numériques
</span>

In [ ]:
v_raw_string = client.query_df(f"""
        SELECT p.device_id, p.payload_timestamp, p.payload_value
        from {releves} p
        WHERE p.payload_metric_id = '{V_brut_pub}'
            AND p.payload_timestamp > toDateTime('2024-01-01 12:00:00')
            AND match (p.payload_value, '(?i)[a-z]')
        settings max_rows_to_read = {maxi}""")
v_raw_string

In [ ]:
# Données des PCE ayant mis de côté les valeurs erronées
v_raw_df = client.query_df(f"""
        SELECT p.device_id, p.payload_timestamp, p.payload_value
        from {releves} p
        WHERE p.payload_metric_id = '{V_brut_pub}'
            AND p.payload_timestamp > toDateTime('2024-01-01 12:00:00')
            AND NOT match (p.payload_value, '(?i)[a-z]')
        settings max_rows_to_read = {maxi}""")
# Merge avec la frequence 
v_raw_df = v_raw_df.merge(dev_freq[['device_id', 'frequence']], on = 'device_id')
v_raw_df['payload_value'] = v_raw_df.payload_value.apply(int)

v_raw_grp = v_raw_df.groupby(['device_id', 'frequence'], as_index = False).agg(
    start=("payload_timestamp", "min"),
    end=("payload_timestamp", "max"),
    nbPay=("payload_value", "count")).sort_values('nbPay')

v_raw_grp["nbTheorique"] = 1
for f, s in v_raw_grp.groupby('frequence'):
    if f == 'J':
        v_raw_grp.loc[s.index, "nbTheorique"] = (s.end - s.start).dt.days.apply(int)
    else:
        v_raw_grp.loc[s.index, "nbTheorique"] = ((s.end - s.start).dt.days/30).apply(int)

<span style="color:orange"> 
Recherche des outliers
</span>

In [ ]:
v_raw_df["outlier"] = False
for d, s in v_raw_df.groupby('device_id'):
    v_raw_df.loc[s.index, "outlier"] = detect_outliers_iqr(s.payload_value.values)

v_raw_df.groupby('device_id').outlier.sum().sort_values()

In [ ]:
s = v_raw_df[v_raw_df.device_id=='65e712f7fde8720373332218'].sort_values('payload_timestamp')
anom = s[s.outlier]
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=s.payload_timestamp, y=s.payload_value,
        marker={'color': red}
    )
)
fig.add_trace(
    go.Scatter(name="Anomalie", x=anom.payload_timestamp, y=anom.payload_value,
                             mode="markers", marker={'color': black, 'size': 4})
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Variation sur le temps du volume brut publié pour le PCE 65e712f7fde8720373332218",
                  xaxis_title="Temps", 
                  yaxis_title="Volume brut en m3",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

<span style="color:orange"> 
Outliers chez les équipements dont la fréquence est 6M
</span>

In [ ]:
v_raw_df[v_raw_df.frequence=='6M'].groupby('device_id').outlier.sum().sort_values()

In [ ]:
s = v_raw_df[v_raw_df.device_id=='655ce5a5359a2473dcd6951b'].sort_values('payload_timestamp')
anom = s[s.outlier]
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=s.payload_timestamp, y=s.payload_value,
        marker={'color': red}
    )
)
fig.add_trace(
    go.Scatter(name="Anomalie", x=anom.payload_timestamp, y=anom.payload_value,
                             mode="markers", marker={'color': black, 'size': 4})
)
fig.update_layout(title=f"Variation sur le temps du volume brut publié pour le PCE 65e712f7fde8720373332218",
                  xaxis_title="Temps", 
                  yaxis_title="Volume brut en m3",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### <span style="color:blue"> b)- Synthèse du nombre d'outliers par PCE ; capteur liés à la consommation et fréquence </span>

In [ ]:
map_capteurs = {"Volume_reel_info":'64f9dc4e964b1c6b3092aebe', "Volume_reel_pub" : '62447752f340cc25cbd2afb0', 
"Volume_brut_info" : '64f9dbc06bd36472e445b9c7', "Volume_brut_pub" : '62447746f340cc25cbd2afaf', "index_reel_info" : '64f9dad0bfcf77e14e7ab3bf',
"index_reel_pub" : '62447396f340cc25cbd2afa2', "index_brut_info" : 'INFORM_RAW_INDEX_VALUE', "index_brut_pub" : '62447388f340cc25cbd2afa1'}

In [ ]:
synth_table = []
for cle in map_capteurs:
    capt = map_capteurs[cle]
    # Données des PCE ayant mis de côté les valeurs erronées
    v_raw_df = client.query_df(f"""
            SELECT p.device_id, p.payload_timestamp, p.payload_value
            from {releves} p
            WHERE p.payload_metric_id = '{capt}'
                AND p.payload_timestamp > toDateTime('2024-01-01 12:00:00')
                AND NOT match (p.payload_value, '(?i)[a-z]')
            settings max_rows_to_read = {maxi}""")
    
    # Merge avec la frequence
    v_raw_df = v_raw_df.merge(dev_freq[['device_id', 'frequency']], on = 'device_id')
    v_raw_df['payload_value'] = v_raw_df.payload_value.apply(float)
    
    v_raw_grp = v_raw_df.groupby(['device_id', 'frequency'], as_index = False).agg(
        start=("payload_timestamp", "min"),
        end=("payload_timestamp", "max"),
        nbPay=("payload_value", "count")).sort_values('nbPay')
    
    v_raw_grp["nbTheorique"] = 1
    for f, s in v_raw_grp.groupby('frequency'):
        if f == 'J':
            v_raw_grp.loc[s.index, "nbTheorique"] = (s.end - s.start).dt.days.apply(int)
        elif f == '6M':
            v_raw_grp.loc[s.index, "nbTheorique"] = ((s.end - s.start).dt.days/30/2).apply(int)
        elif f == 'M':
            v_raw_grp.loc[s.index, "nbTheorique"] = ((s.end - s.start).dt.days/30).apply(int)
    v_raw_grp['nb_missing'] = v_raw_grp.nbTheorique - v_raw_grp.nbPay
    
    ## outliers 
    v_raw_df["outlier"] = False
    for d, s in v_raw_df.groupby('device_id'):
        v_raw_df.loc[s.index, "outlier"] = detect_outliers_iqr(s.payload_value.values)
    v_raw_outliers = v_raw_df.groupby('device_id', group_keys=False).outlier.sum().sort_values()
    v_raw_grp = v_raw_grp.merge(pd.DataFrame(v_raw_outliers), on='device_id')
    v_raw_grp = v_raw_grp.drop('device_id', axis=1).groupby('frequency').mean()
    v_raw_grp.loc[:,'capteur_name'] = cle
    v_raw_grp.loc[:,'capteur'] = capt
    synth_table.append(v_raw_grp)
synth_table = pd.concat(synth_table)
synth_table['type'] = synth_table.capteur_name.apply(lambda x: x.split('_')[-1])
synth_table['data'] = synth_table.capteur_name.apply(lambda x: x.split('_')[0])
synth_table['mesure'] = synth_table.capteur_name.apply(lambda x: x.split('_')[-1])

In [ ]:
synth_table.reset_index().drop(['capteur_name', 'capteur'], axis=1).groupby(['frequency', 'type', 'data', 'mesure']).mean()

### <span style="color:blue"> c)- Couverture de la continuité des données *PUBLIÉES* par PCE </span>

In [ ]:
def tag_continuous_seq(smp, first_id, grp_by, freq):  
    tags_df = []    
    s_cnt = first_id
    for idx, s in smp.groupby(grp_by):
        s = s.sort_values('payload_timestamp').reset_index(drop=True)
        delta_s = pd.DataFrame(s.payload_timestamp.diff().rename('delta'))
        missed = s[delta_s.delta > dt.timedelta(seconds=freq)]
        t0 = s.index[0]
        if missed.empty:
            s[["s_id"]] = s_cnt
            s_cnt += 1
        else:
            for t1, row in missed.iterrows():
                s.loc[t0:t1-1, "s_id"] = s_cnt
                s_cnt +=1
                t0 = t1        
            s.loc[t0:, "s_id"] = s_cnt
            s_cnt +=1
        tags_df.append(s)
    return pd.concat(tags_df)

In [ ]:
sample_df = []
for cle in map_capteurs:
    capt = map_capteurs[cle]
    sample = client.query_df(f"""
            SELECT p.device_id, p.payload_timestamp, p.payload_value
            from {releves} p
            WHERE p.payload_metric_id = '{capt}'
                AND p.payload_timestamp > toDateTime('2024-01-01 12:00:00')
                AND NOT match (p.payload_value, '(?i)[a-z]')
            settings max_rows_to_read = {maxi}""")
    sample.loc[:,'capteur_name'] = cle
    sample.loc[:,'capteur'] = capt
    sample_df.append(sample)
sample_df = pd.concat(sample_df)
sample_df = sample_df.merge(dev_freq[['device_id', 'frequency']], on = 'device_id')
sample_df['payload_value'] = sample_df.payload_value.apply(float)

In [ ]:
seq_df = []
for freq, smp in sample_df.groupby('frequency'):
    if freq == '':
        continue
    match freq:
        case 'J':
            delta = 60*60*24
        case 'M':
            delta = 60*60*24*30
        case '6M':
            delta =  60*60*24*30.5*6
    smp = tag_continuous_seq(smp, 0, ['device_id', 'capteur_name', 'capteur'], delta)
    seq_df.append(smp)
seq_df = pd.concat(seq_df)
sequences = seq_df.groupby(['frequency','capteur_name', 'device_id', 's_id'], as_index=False).agg(
    couverture=('payload_timestamp', lambda x: x.max() - x.min())
)

In [ ]:
sequences['type'] = sequences.capteur_name.apply(lambda x: x.split('_')[-1])

In [ ]:
max_val = sequences[sequences['type']=='pub'].couverture.max()
tickvals = [0, 60*60*24,
            86400*30, 2592000*6, 2592000*12, 37576200]
ticktext = ['< 1 journée', '(1j - 1 mois])', '(1 mois - 6 mois])', '(6 mois - 1 an])', 
            '> 1 ans']
sequences["interval"] = pd.cut(sequences["couverture"].apply(lambda x: x.total_seconds()), bins=tickvals, labels=ticktext)
count_df = sequences[sequences['type']=='pub'].groupby(['frequency','interval'], as_index=False).count()

fig = go.Figure()
for freq, s in count_df.groupby('frequency'):
    fig.add_trace(
        go.Bar(name = freq, x=s.interval, y=s.couverture
        )
    )
fig.update_yaxes(type="log")
fig.update_layout(title=f"Couverture des relevés publiés par séquence continue de données",
                  xaxis_title="Couverture", 
                  yaxis_title="Nombre de séquences continues",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
max_val = sequences[sequences['type']=='info'].couverture.max()
tickvals = [0, 60*60*24,
            86400*30, 2592000*6, 2592000*12, 37576200]
ticktext = ['< 1 journée', '(1j - 1 mois])', '(1 mois - 6 mois])', '(6 mois - 1 an])', 
            '> 1 ans']
sequences["interval"] = pd.cut(sequences["couverture"].apply(lambda x: x.total_seconds()), bins=tickvals, labels=ticktext)
count_df = sequences[sequences['type']=='info'].groupby(['frequency','interval'], as_index=False).count()

fig = go.Figure()
for freq, s in count_df.groupby('frequency'):
    fig.add_trace(
        go.Bar(name = freq, x=s.interval, y=s.couverture
        )
    )
fig.update_yaxes(type="log")
fig.update_layout(title=f"Couverture des relevés informatifs par séquence continue de données",
                  xaxis_title="Couverture", 
                  yaxis_title="Nombre de séquences continues",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### <span style="color:blue"> d)-Durée des périodes de données manquantes par PCE </span>

In [ ]:
whole_df = seq_df.groupby(['frequency', 'capteur_name', 'device_id', 's_id'], as_index=False).payload_timestamp.agg(['min', 'max'])
whole_length_df = []
for key, grp in whole_df.groupby(['frequency', 'capteur_name', 'device_id']):
    t0 = grp['max'].values[0]
    if grp.shape[0]<2:
        continue
    for i, row in grp.iloc[1:].iterrows():
        t1 = row['min']
        whole_length_df.append({
            "frequency":key[0], "capteur_name":key[1], "device_id":key[2], "s_id":row.s_id, "length":t1-t0})
        t0 = row['max']
whole_length_df = pd.DataFrame(whole_length_df)
whole_length_df['type'] = whole_length_df.capteur_name.apply(lambda x: x.split('_')[-1])

In [ ]:
max_val = whole_length_df[whole_length_df['type']=='pub']["length"].max()
tickvals = [0, 60*60*24,
            86400*30, 2592000*6, 2592000*12, 37576200]
ticktext = ['< 1 journée', '(1j - 1 mois])', '(1 mois - 6 mois])', '(6 mois - 1 an])', '> 1 ans']
whole_length_df["interval"] = pd.cut(whole_length_df["length"].apply(lambda x: x.total_seconds()), bins=tickvals, labels=ticktext)
count_df = whole_length_df[whole_length_df['type']=='pub'].groupby(['frequency','interval'], as_index=False).count()

fig = go.Figure()
for freq, s in count_df.groupby('frequency'):
    fig.add_trace(
        go.Bar(name = freq, x=s.interval, y=s.length
        )
    )
fig.update_yaxes(type="log")
fig.update_layout(title=f"Durée des périodes sans donnée publiée par équipement et capteur",
                  xaxis_title="Durée de période", 
                  yaxis_title="Nombre de couples équipement-capteurs affectés",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
max_val = whole_length_df[whole_length_df['type']=='info']["length"].max()
tickvals = [0, 60*60*24,
            86400*30, 2592000*6, 2592000*12, 37576200]
ticktext = ['< 1 journée', '(1j - 1 mois])', '(1 mois - 6 mois])', '(6 mois - 1 an])', '> 1 ans']
whole_length_df["interval"] = pd.cut(whole_length_df["length"].apply(lambda x: x.total_seconds()), bins=tickvals, labels=ticktext)
count_df = whole_length_df[whole_length_df['type']=='info'].groupby(['frequency','interval'], as_index=False).count()

fig = go.Figure()
for freq, s in count_df.groupby('frequency'):
    fig.add_trace(
        go.Bar(name = freq, x=s.interval, y=s.length
        )
    )
fig.update_yaxes(type="log")
fig.update_layout(title=f"Durée des périodes sans données informatives par équipement et capteur",
                  xaxis_title="Durée de période", 
                  yaxis_title="Nombre de couples équipement-capteurs affectés",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### <span style="color:blue"> e)-Nombre de périodes manquantes par PCE </span>

In [ ]:
nb_wholes = whole_df.groupby(['frequency', 'capteur_name', 'device_id'], as_index=False).s_id.apply(lambda x: x.count()-1)
nb_wholes = nb_wholes[nb_wholes.s_id>0]

In [ ]:
fig = go.Figure()
for freq, s in nb_wholes.groupby('frequency'):
    fig.add_trace(
        go.Histogram(name=freq, x=s.s_id
        )
    )
fig.update_xaxes(type="log")
fig.update_yaxes(type="log")
fig.update_layout(title=f"Nombre de périodes sans données par équipement et capteur",
                  xaxis_title="Nombre de périodes", 
                  yaxis_title="Nombre de couples équipement-capteurs affectés",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

v_raw_df = client.query_df(f"""
        SELECT p.device_id, p.payload_timestamp, p.payload_value
        from {releves} p
        WHERE p.payload_metric_id = '{V_brut_pub}'
            AND p.payload_timestamp > toDateTime('2024-01-01 12:00:00')
            AND NOT match (p.payload_value, '(?i)[a-z]')
        settings max_rows_to_read = {maxi}""")

In [ ]:
idx_col = ['frequency', 'capteur_name', 'device_id']
synthese_df = sequences.groupby(idx_col, as_index=False).couverture.mean()\
    .merge(whole_length_df.groupby(idx_col, as_index=False).length.mean()\
           .rename({'length':'avg_len_trous'}, axis=1), on=idx_col)\
    .merge(nb_wholes.groupby(idx_col, as_index=False).s_id.mean()\
       .rename({'s_id':'avg_nb_trous'}, axis=1), on=idx_col)
synthese_df

In [ ]:
fig = go.Figure()
for freq, s in synthese_df.groupby('frequency'):
    fig.add_trace(go.Scatter3d(name = freq, x=s.couverture, y=s.avg_len_trous, z=s.avg_nb_trous, mode='markers'))
fig.update_layout(title=f"Synthèse de la couverture et données manquante par PCE et par fréquence",
                  xaxis_title="Couverture moyenne continuité de données.", 
                  yaxis_title="Durée moyenne des périodes manquantes",
                  # zaxis_title="Nombre moyen de périodes manquantes",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=800,
                template="simple_white")
fig.show()

### <span style="color:blue"> e)- Valeurs autres que numériques </span>

In [ ]:
pub_capteurs = {"Volume_reel_pub" : '62447752f340cc25cbd2afb0',"Volume_brut_pub" : '62447746f340cc25cbd2afaf', "index_reel_info" : '64f9dad0bfcf77e14e7ab3bf',
"index_reel_pub" : '62447396f340cc25cbd2afa2', "index_brut_info" : 'INFORM_RAW_INDEX_VALUE', "index_brut_pub" : '62447388f340cc25cbd2afa1'}
str_df = []
for cle in pub_capteurs:
    capt = pub_capteurs[cle]
    str_values = client.query_df(f"""
        SELECT p.device_id, p.payload_timestamp, p.payload_value
        from {releves} p
        WHERE p.payload_metric_id = '{capt}'
            AND p.payload_timestamp > toDateTime('2024-01-01 12:00:00')
            AND match (p.payload_value, '(?i)[a-z]')
        settings max_rows_to_read = {maxi}""")
    str_df.append(str_values)

In [ ]:
pd.concat(str_df)

### <span style="color:blue"> f)- Overlapping mesures </span>

In [ ]:
# données publiées
pub_capteurs = {"Volume_reel_pub" : '62447752f340cc25cbd2afb0',"Volume_brut_pub" : '62447746f340cc25cbd2afaf', 
                "index_reel_pub" : '62447396f340cc25cbd2afa2', "index_brut_pub" : '62447388f340cc25cbd2afa1', 'date_fin_pub': 'CONSUMPTION_END_DATE', 
                'date_debut_pub':'CONSUMPTION_START_DATE'}
sens_data = []
for cle in pub_capteurs:
    capt = pub_capteurs[cle]
    df_values = client.query_df(f"""
        SELECT p.payload_metric_id, p.device_id, p.payload_timestamp, p.payload_value
        from {releves} p
        WHERE p.payload_metric_id = '{capt}'
            AND p.payload_timestamp > toDateTime('2024-01-01 12:00:00')
        settings max_rows_to_read = {maxi}""")
    sens_data.append(df_values)
# sens_data = pd.concat(sens_data)
doublons = sens_data.groupby(['payload_metric_id', 'device_id', 'payload_timestamp']).count().sort_values('payload_value')
doublons = doublons[doublons.payload_value >1]

#### Doublons sur le payload_timestamp

In [ ]:
doublons

##### Overlapping sur CONSUMPTION_END_DATE et CONSUMPTION_START_DATE

## <span style="color:green"> 4 - Analyse des types de valeurs des relevés, tous capteurs confondus</span>

### <span style="color:blue"> b)-Analyse des types de valeurs des relevés</span>

#### <span style="color:pink"> Chaînes de caractères </span>

In [ ]:
df_cat = client.query_df(f"""
    SELECT device_id, payload_metric_id, payload_timestamp, payload_value
    from {releves} 
    WHERE device_id IN {gaz_device[:5000]}
        AND match(payload_value, '.*[A-Za-z].*[A-Za-z].*') 
        AND payload_timestamp > toDateTime('2024-01-01 12:00:00')
    settings max_rows_to_read = {maxi}""")
df_cat2 = client.query_df(f"""
    SELECT device_id, payload_metric_id, payload_timestamp, payload_value
    from {releves} 
    WHERE device_id IN {gaz_device[5000:]}
        AND match(payload_value, '.*[A-Za-z].*[A-Za-z].*') 
        AND payload_timestamp > toDateTime('2024-01-01 12:00:00')
    settings max_rows_to_read = {maxi}""")

df_cat = pd.concat([df_cat, df_cat2])
nb_categories = pd.DataFrame(df_cat.payload_value.value_counts()).reset_index()

#### <span style="color:pink"> Entiers et nombres flottants </span>

In [ ]:
df_int = client.query_df(f"""
    SELECT device_id, payload_metric_id, payload_timestamp, payload_value
    from {releves} 
    WHERE device_id IN {grdf_devices[:5000]}
        AND (match(payload_value, '^[0-9]+$') OR match(payload_value, '^[0-9]+\\.[0-9]+$'))
        AND payload_timestamp > toDateTime('2024-01-01 12:00:00')
    settings max_rows_to_read = {maxi}""")
df_int2 = client.query_df(f"""
    SELECT device_id, payload_metric_id, payload_timestamp, payload_value
    from {releves} 
    WHERE device_id IN {grdf_devices[5000:]}
        AND (match(payload_value, '^[0-9]+$') OR match(payload_value, '^[0-9]+\\.[0-9]+$'))
        AND payload_timestamp > toDateTime('2024-01-01 12:00:00')
    settings max_rows_to_read = {maxi}""")
df_int = pd.concat([df_int, df_int2])

nb_categories = pd.concat([nb_categories, 
                           pd.DataFrame([["Valeurs numériques", df_int.shape[0]]], 
                                        columns=['payload_value','count'])])

#### <span style="color:pink"> Timestamp & datetime</span>

In [ ]:
df_dat = client.query_df(f"""
    SELECT device_id, payload_metric_id, payload_timestamp, payload_value
    from {releves} 
    WHERE device_id IN {grdf_devices[:5000]}
        AND match(payload_value, '^.*[0-9]+-[0-9]+.*$')
        AND payload_timestamp > toDateTime('2024-01-01 12:00:00')
    settings max_rows_to_read = {maxi}""")
df_dat2 = client.query_df(f"""
    SELECT device_id, payload_metric_id, payload_timestamp, payload_value
    from {releves} 
    WHERE device_id IN {grdf_devices[5000:]}
        AND match(payload_value, '^.*[0-9]+-[0-9]+.*$')
        AND payload_timestamp > toDateTime('2024-01-01 12:00:00')
    settings max_rows_to_read = {maxi}""")
df_dat = pd.concat([df_dat, df_dat2])

nb_categories = pd.concat([nb_categories,
                           pd.DataFrame([["Timestamp", df_dat.shape[0]]],
                                        columns=['payload_value','count'])])

### <span style="color:blue"> c)-Synthèse de la distribution des relevés par type de valeur</span>

In [ ]:
nb_categories.sort_values('count', inplace=True)

fig = go.Figure()
fig.add_trace(
    go.Bar(x=nb_categories.payload_value, y=nb_categories['count'], text=nb_categories['count'],
        marker={'color': orange}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Répartition des valeurs des relevés GRDF (de 01-2024 à 09-2025)",
                  xaxis_title="Catégorie", 
                  yaxis_title="Nombre de relevés",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### <span style="color:blue"> d)-Analyse des valeurs entières et nombres flottants</span>

In [ ]:
print("Nombre d'équipements :", len(df_int.device_id.unique()))
print("Nombre de capteurs: ", len(df_int.payload_metric_id.unique()))
df_int['payload_value'] = df_int.payload_value.map(float)

In [ ]:
metrics = (df_int.groupby("payload_metric_id")["payload_value"]
                    .agg(avg="mean", min="min", max="max", med="median",
                        q1=lambda x: x.quantile(0.25),
                        q3=lambda x: x.quantile(0.75),
                        nb="count").reset_index())

In [ ]:
fig = go.Figure()
fig.add_trace(go.Box(x=metrics.index, q1=metrics.q1.values, median=metrics['med'].values,
                      q3=metrics['q3'].values, lowerfence=metrics['min'].values,
                      upperfence=metrics['max'].values, mean=metrics['avg'].values, 
                    marker={'color':pink}, whiskerwidth=1, marker_size=2, line_width=1))
fig.add_trace(go.Bar(x=metrics.index, y=metrics.nb, marker={'color':blue, 'opacity':0.5}, text=metrics.nb))
fig.update_yaxes(type='log', range = (0.00001,10))
fig.update_layout(title=f"Répartition des valeurs des relevés GRDF par capteur (de 01-2024 à 09-2025) <br>(Nombre de relevés en bleu)",
                  xaxis_title="Capteur", 
                  yaxis_title="Puissance relevée <br> (Echelle logarithmique)",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1300, height=400,
                template="simple_white", showlegend=False)
fig.show()

## <span style="color:green"> 5 - Description des fréquences des relevés pour les capteurs liés aux équipements de la famille 'Compteur Gaz' </span> 

In [ ]:
# avg(toInt8(substring(toString(p.payload_timestamp), 18, 2)) # pour calculer la valeur moyenne du chiffre des secondes
df_freq = client.query_df(f"""
    SELECT p.device_id, p.payload_metric_id, arrayReduce('avg', diffs) AS avgDiff 
    FROM( 
        SELECT 
            p.device_id, 
            p.payload_metric_id, 
            arrayDifference(arraySort(groupArray(p.payload_timestamp))) AS diffs
        FROM {releves} p
        INNER JOIN {equipements} d 
        ON p.device_id = d.device_id
        WHERE d.device_type = 'Compteur Gaz' AND p.payload_source = '{source}'
    GROUP BY 
        p.device_id, 
        p.payload_metric_id) p
    ORDER BY avgDiff DESC
    settings max_rows_to_read = {maxi}""")

In [ ]:
df_freq['timedelta'] = df_freq.avgDiff.apply(lambda x: dt.timedelta(seconds=x))
max_val = df_freq.head(100).avgDiff.max()
tickvals = [-float('inf'), 1, 60, 
            60*5, 60*10, 60*30, 60*45, 
            3600, 3600*6, 3600*12, 
            86400, 86400*5, 86400*15, 86400*30, max_val]
ticktext = ['< 1s', '< 1mn', '(1mn - 5mn])', '(5mn - 10mn])', '(10mn - 30mn])', 
           '(30mn - 45mn])', '(45mn - 1h])', '(1h - 6h])', '(6h - 12h])', 
           '(12h - 1jour])', '(1jour - 5jours])', '(5jours - 15jours])', 
           '(15jours - 30jours])', '> 1mois']
df_freq["interval"] = pd.cut(df_freq["avgDiff"], bins=tickvals, labels=ticktext)
counts = df_freq["interval"].value_counts(sort=False)

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': pink}
    )
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Fréquence moyenne des relevés par couples équipement-capteur",
                  xaxis_title="fréquence moyenne", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

La même chose mais en considérant seulement les 20 derniers relevés

In [ ]:
# avg(toInt8(substring(toString(p.payload_timestamp), 18, 2)) # pour calculer la valeur moyenne du chiffre des secondes
df_freq = client.query_df(f"""
    SELECT 
        p.device_id, 
        p.payload_metric_id, 
        arrayReduce('avg', diffs) AS avgDiff
    FROM (
        SELECT 
            device_id, 
            payload_metric_id, 
            arrayDifference(arraySort(groupArray(payload_timestamp))) AS diffs
        FROM (
            SELECT 
                p.device_id, 
                p.payload_metric_id, 
                p.payload_timestamp
            FROM {releves} p
            INNER JOIN {equipements} d 
                ON p.device_id = d.device_id
            WHERE d.device_type = 'Compteur Gaz' AND p.payload_source = '{source}'
            QUALIFY row_number() OVER (PARTITION BY p.device_id, p.payload_metric_id ORDER BY p.payload_timestamp DESC) <= 20
        )
        GROUP BY device_id, payload_metric_id
    ) p
    ORDER BY avgDiff DESC
    SETTINGS max_rows_to_read = {maxi};""")

In [ ]:
df_freq['timedelta'] = df_freq.avgDiff.apply(lambda x: dt.timedelta(seconds=x))
max_val = df_freq.head(100).avgDiff.max()
tickvals = [-float('inf'), 1, 60, 
            60*5, 60*10, 60*30, 60*45, 
            3600, 3600*6, 3600*12, 
            86400, 86400*5, 86400*15, 86400*30, max_val]
ticktext = ['< 1s', '< 1mn', '(1mn - 5mn]', '(5mn - 10mn]', '(10mn - 30mn]', 
           '(30mn - 45mn]', '(45mn - 1h]', '(1h - 6h]', '(6h - 12h]', 
           '(12h - 1jour]', '(1jour - 5jours]', '(5jours - 15jours]', 
           '(15jours - 30jours]', '> 1mois']
df_freq["interval"] = pd.cut(df_freq["avgDiff"], bins=tickvals, labels=ticktext)
counts = df_freq["interval"].value_counts(sort=False)

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': blue}
    )
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Fréquence moyenne des relevés par couples équipement-capteur",
                  xaxis_title="fréquence moyenne", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### Relevés dont la fréquence ne peut être estimée

In [ ]:
no_freq_data = []
for dev, s in df_freq[df_freq.avgDiff <1].groupby('device_id'):
    sens_l = s.payload_metric_id.to_list()
    q = f"""
    SELECT device_id, payload_metric_id, payload_timestamp, payload_value
    FROM datawarehouse_prod.fact_payload 
    WHERE device_id='{dev}' AND payload_metric_id IN {sens_l}
    settings max_rows_to_read = {maxi};"""
    no_freq_data.append(client.query_df(q))
no_freq_data = pd.concat(no_freq_data)

In [ ]:
no_freq_data

<span style="color:orange">  Il n'y a aucun doublon quelle que soit la famille d'équipement </span>

## <span style="color:green"> 6 - Description de la couverture des relevés pour les capteurs liés aux équipements de la famille 'Compteur Gaz' et dont les données sont de type int - float </span>

### <span style="color:blue"> a)- Relevés à la journée (échantillon de 100 combinaisons équipement - capteur)</span>

In [ ]:
df_freq[df_freq.timedelta == dt.timedelta(days=1)]

## <span style="color:green"> 7 - Délai moyen d'ingestion de la donnée </span>

### <span style="color:blue"> a) - pour toutes les données d'un équipement </span>

In [ ]:
df_lag = client.query_df(f"""
    SELECT d.device_type, p.device_id, p.payload_metric_id, avg(p.ingested_at - p.payload_timestamp) AS avgLag 
    FROM {releves} p
    INNER JOIN {equipements} d 
        ON p.device_id = d.device_id
    WHERE p.payload_source = '{source}'
    GROUP BY 
        d.device_type, p.device_id, p.payload_metric_id
    ORDER BY avgLag DESC
    settings max_rows_to_read = {maxi}""")     
df_lag

In [ ]:
df_lag['timedelta'] = df_lag.avgLag.apply(lambda x: dt.timedelta(seconds=x))
max_val = df_lag.avgLag.max()
tickvals = [86400, 86400*5, 86400*15, 86400*30, max_val]
ticktext = ['(1jour - 5jours])', '(5jours - 15jours])', 
           '(15jours - 30jours])', '> 1mois']
df_lag["interval"] = pd.cut(df_lag["avgLag"], bins=tickvals, labels=ticktext)
counts = df_lag["interval"].value_counts(sort=False)

fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': blue}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Délai moyen d'ingestion des relevés par couple 'equipement-capteur'",
                  xaxis_title="Délai moyen", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### <span style="color:blue"> b) - En considérant le relevé le plus ancien ingéré </span>

In [ ]:
df_lag = client.query_df(f"""
    SELECT
    p.device_id,
    p.payload_metric_id,
    max(p.ingested_at - p.payload_timestamp) AS lag,
    min(p.ingested_at) MES
FROM {releves} p
WHERE p.payload_source = '{source}'
GROUP BY 
    p.device_id,
    p.payload_metric_id
ORDER BY MES DESC
SETTINGS max_rows_to_read = {maxi};""")     
df_lag

In [ ]:
df_lag['timedelta'] = df_lag.lag.apply(lambda x: dt.timedelta(seconds=x))
max_val = df_lag.lag.max()
tickvals = [86400, 86400*5, 86400*15, 86400*30, 
            2592000*6, 2592000*12, 31104000*2, 
            31104000*3, 31104000*4, 31104000*5, max_val]
ticktext = ['(1jour - 5jours])', '(5jours - 15jours])', 
           '(15jours - 30jours])', '(1mois - 6mois]', 
            '(6mois - 1 an]', '(1an - 2ans]', '(2mois - 3ans]', 
           '(3ans - 4ans]', '(4ans - 5ans]', '>5ans']
df_lag["interval"] = pd.cut(df_lag["lag"], bins=tickvals, labels=ticktext)
counts = df_lag["interval"].value_counts(sort=False)

fig = go.Figure()
fig.add_trace(
    go.Bar(x=counts.index, y=counts,
        marker={'color': blue}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Délai maximal d'ingestion des relevés par couple 'equipement-capteur'",
                  xaxis_title="Délai moyen", 
                  yaxis_title="Nombre de couples <br> equipement-capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

## <span style="color:green"> 8 - Différents types de dates </span>

In [ ]:
df = client.query_df(f"""
SELECT payload_metric_id, payload_value, device_id
FROM {releves}
WHERE payload_source = '{source}'
AND match(payload_metric_id, '((?i)^CONSUMPTION.*DATE$|^DATE_CONSOMMATION.*)')
settings max_rows_to_read = {maxi};""")
df["datetime"] = pd.to_datetime(df.payload_value.apply(lambda x: x[1:-1]), utc=True)

In [ ]:
group = df.groupby(['device_id', 'payload_metric_id'], group_keys=False).count()
group = group.pivot_table(columns="payload_metric_id", index="device_id", values="payload_value")
group

In [ ]:
group.isna().sum()

In [ ]:
device = 
s = client.query_df(f"""
SELECT * 
FROM {releves}
WHERE device_id = '65fb47456b74ca736521fb7b'
	AND payload_timestamp > toDateTime('2025-03-01 00:00:00')
	AND payload_timestamp < toDateTime('2025-04-01 23:00:00')
ORDER BY payload_timestamp
settings max_rows_to_read={maxi};""")

In [ ]:
color_map = {'CONSUMPTION_END_DATE':0.5, 'CONSUMPTION_START_DATE':1, 'DATE_CONSOMMATION_DEBUT':1.5, 'DATE_CONSOMMATION_FIN':2}
metric_color = s['payload_metric_id'].map(color_map)

y_labels = color_map.keys()
fig = go.Figure()
fig.add_trace(go.Scatter(x=s['datetime'], mode='markers', y=metric_color))
fig.update_layout(title=f"Dates auxquelle des relevés apparten",
                  xaxis_title="Délai moyen", 
                  yaxis=dict(
                        tickmode='array',                     # nécessaire pour utiliser tickvals/ticktext
                        tickvals=list(color_map.values()),    # positions des ticks
                        ticktext=list(y_labels)               # labels correspondants
                    ),
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

## <span style="color:green"> 9 - Données chevauchantes </span>

In [ ]:
# Capteurs de consommation
V_brut_pub = '62447746f340cc25cbd2afaf'
date_fin_pub = 'CONSUMPTION_END_DATE'
date_debut_pub = 'CONSUMPTION_START_DATE'
coeff_PTA = '624477c5f340cc25cbd2afb1'
coef_PCS = '624477e1f340cc25cbd2afb2'
raison = '624476dbf340cc25cbd2afae'
coef = '624477f0f340cc25cbd2afb3'

In [ ]:
df = client.query_df(f"""
SELECT payload_metric_id, payload_value, payload_timestamp, device_id
FROM {releves}
WHERE payload_source = '{source}'
AND payload_metric_id IN {[V_brut_pub, date_fin_pub, date_debut_pub, coeff_PTA, coef_PCS, raison]}
AND payload_timestamp > toDateTime('2024-01-01 00:00:00')
settings max_rows_to_read = {maxi};""")

In [ ]:
df2 = client.query_df(f"""
SELECT payload_metric_id, payload_value, payload_timestamp, device_id
FROM {releves}
WHERE payload_source = '{source}'
AND payload_metric_id IN {[V_brut_pub, date_fin_pub, date_debut_pub, coef, raison]}
AND payload_timestamp > toDateTime('2024-01-01 00:00:00')
settings max_rows_to_read = {maxi};""")

In [ ]:
dev_freq = client.query_df(f"""
        SELECT DISTINCT device_id, frequency, grdf_frequency
        FROM {grdf_all}
        settings max_rows_to_read = {maxi}""")

In [ ]:
df_cross2 = df2.pivot_table(index = ['device_id','payload_timestamp'], columns = "payload_metric_id", values="payload_value", aggfunc='first')
df_cross2 = df_cross2.reset_index()
df_cross2['CONSUMPTION_END_DATE'] = pd.to_datetime(df_cross2['CONSUMPTION_END_DATE'].apply(lambda x: x[1:11] if pd.notnull(x) else x))
df_cross2['CONSUMPTION_START_DATE'] = pd.to_datetime(df_cross2['CONSUMPTION_START_DATE'].apply(lambda x: x[1:11] if pd.notnull(x) else x))
df_cross2 = df_cross2.merge(dev_freq[['device_id', 'grdf_frequency']])

In [ ]:
df_cross = df.pivot_table(index = ['device_id','payload_timestamp'], columns = "payload_metric_id", values="payload_value", aggfunc='first')
df_cross = df_cross.reset_index()
df_cross['CONSUMPTION_END_DATE'] = pd.to_datetime(df_cross['CONSUMPTION_END_DATE'].apply(lambda x: x[1:11] if pd.notnull(x) else x))
df_cross['CONSUMPTION_START_DATE'] = pd.to_datetime(df_cross['CONSUMPTION_START_DATE'].apply(lambda x: x[1:11] if pd.notnull(x) else x))
df_cross = df_cross.merge(dev_freq[['device_id', 'grdf_frequency']])

In [ ]:
df_cross.sort_values(['device_id', 'payload_timestamp'])

In [ ]:
df_cross[df_cross.CONSUMPTION_START_DATE == df_cross.CONSUMPTION_END_DATE]['624476dbf340cc25cbd2afae'].value_counts()

In [ ]:
df_cross[(df_cross['624476dbf340cc25cbd2afae']=='"RNO"') & (df_cross.grdf_frequency=='1M')].device_id.value_counts().mean()

In [ ]:
df_cross2

In [ ]:
V_brut_pub = '62447746f340cc25cbd2afaf'
date_fin_pub = 'CONSUMPTION_END_DATE'
date_debut_pub = 'CONSUMPTION_START_DATE'
coeff_PTA = '624477c5f340cc25cbd2afb1'
coef_PCS = '624477e1f340cc25cbd2afb2'
raison = '624476dbf340cc25cbd2afae'
coef = '624477f0f340cc25cbd2afb3'

In [ ]:
df_cross2[(df_cross2['624476dbf340cc25cbd2afae']=='"RNO"') & (df_cross.grdf_frequency=='1M') & (df_cross2['CONSUMPTION_START_DATE'].isna())]

In [ ]:
print(df_cross2[(df_cross2['624476dbf340cc25cbd2afae']=='"RNO"') & (df_cross.grdf_frequency=='1M')].isna()['62447746f340cc25cbd2afaf'].sum()/263781)
print(df_cross2[(df_cross2['624476dbf340cc25cbd2afae']=='"RNO"') & (df_cross.grdf_frequency=='1M')].isna()['CONSUMPTION_END_DATE'].sum()/263781)
print(df_cross2[(df_cross2['624476dbf340cc25cbd2afae']=='"RNO"') & (df_cross.grdf_frequency=='1M')].isna()['CONSUMPTION_START_DATE'].sum()/263781)
print(df_cross2[(df_cross2['624476dbf340cc25cbd2afae']=='"RNO"') & (df_cross.grdf_frequency=='1M')].isna()['624477f0f340cc25cbd2afb3'].sum()/263781)

In [ ]:
df_cross2[(df_cross2['624476dbf340cc25cbd2afae']=='"RNO"') & (df_cross.grdf_frequency=='1M')].isna().sum(axis=1).value_counts()/263781

In [ ]:
df_cross[(df_cross['624476dbf340cc25cbd2afae']=='"RNO"') & (df_cross.grdf_frequency=='1M')].isna().sum(axis=1).value_counts()/75532

In [ ]:
205613+36223+15701+6722

In [ ]:
overlap_df = []
for dev, test in df_cross.dropna().groupby('device_id'):
    test = test.sort_values('CONSUMPTION_START_DATE')
    minimum = test.CONSUMPTION_START_DATE.min()
    maximum = test[test.CONSUMPTION_START_DATE == minimum].CONSUMPTION_END_DATE.values[0]
    for i, dt in test.iterrows():
        nb_doub = test[((test.CONSUMPTION_END_DATE < maximum) & (test.CONSUMPTION_END_DATE > minimum))|\
    ((test.CONSUMPTION_START_DATE < maximum) & (test.CONSUMPTION_START_DATE > minimum))].shape[0]
        if nb_doub > 0:
            overlap_df.append({
            'device_id':dt.device_id, 'CONSUMPTION_START_DATE': dt.CONSUMPTION_START_DATE, 'CONSUMPTION_END_DATE':dt.CONSUMPTION_START_DATE, \
            'overlap_lower':minimum, 'overlap_upper':maximum, 'frequency':dt.grdf_frequency})
        minimum = dt.CONSUMPTION_START_DATE
        maximum = dt.CONSUMPTION_END_DATE
overlap_df = pd.DataFrame(overlap_df)

In [ ]:
overlap_df

In [ ]:
overlap_df[overlap_df.CONSUMPTION_START_DATE != overlap_df.CONSUMPTION_END_DATE]

Pas de d'overlap entre les dates

# <span style="color:purple"> 3 - Recherche d'anomalies parmi les données publiées </span> 

In [ ]:
def detect_outliers_iqr(data, k = 2.5):
    q25, q75 = np.percentile(data, [25, 75])
    iqr = q75 - q25
    cutoff = iqr * k
    thres = q75 + cutoff
    return data > thres

In [ ]:
# Capteurs de consommation
V_brut_pub = '62447746f340cc25cbd2afaf'

date_fin_pub = 'CONSUMPTION_END_DATE'
date_debut_pub = 'CONSUMPTION_START_DATE'

raison = '624476dbf340cc25cbd2afae'

In [ ]:
dev_freq = client.query_df(f"""
        SELECT DISTINCT device_id, grdf_frequency as frequency
        FROM {grdf_all}
        settings max_rows_to_read = {maxi}""")

In [ ]:
v_raw_df = client.query_df(f"""
        SELECT p.device_id, p.payload_metric_id, p.payload_timestamp, p.payload_value
        from {releves} p
        WHERE payload_source = '{source}'
            AND p.payload_metric_id IN {[V_brut_pub, raison, date_fin_pub, date_debut_pub]}
            AND p.payload_timestamp > toDateTime('2024-01-01 12:00:00')
        settings max_rows_to_read = {maxi}""")

## <span style="color:blue"> a)- Test avec la méthode de l'écart interquartile </span>

In [ ]:
volume_data = v_raw_df.pivot_table(index = ['device_id','payload_timestamp'], columns = "payload_metric_id", values="payload_value", aggfunc='first')\
    .reset_index().rename({'62447746f340cc25cbd2afaf':'Volume', '624476dbf340cc25cbd2afae':'Raison'}, axis=1)
normal_volume = volume_data[(volume_data.Raison=='"RNO"') & ~(volume_data.Volume.isna())]
normal_volume = normal_volume.merge(dev_freq[['device_id', 'frequency']], on = 'device_id')
normal_volume['Volume'] = normal_volume.Volume.apply(int)

In [ ]:
normal_volume

In [ ]:
normal_volume['last_day'] = normal_volume.CONSUMPTION_END_DATE.apply(lambda x: str(x)[9:11])
normal_volume[normal_volume.last_day!='01'].sort_values('frequency')

In [ ]:
nbPay = normal_volume.groupby(['device_id', 'frequency'], as_index = False).agg(
    start=("payload_timestamp", "min"),
    end=("payload_timestamp", "max"),
    nbPay=("Volume", "count")).sort_values('nbPay')
for d, s in normal_volume.groupby('device_id'):
    normal_volume.loc[s.index, "outlier"] = detect_outliers_iqr(s.Volume.values)
stats_outliers = nbPay.merge(normal_volume.groupby('device_id').outlier.sum(), on = 'device_id')

In [ ]:
stats_outliers

## <span style="color:blue"> b)- Test avec un z-score (ne fonctionne que si les données sont normales) </span>

In [ ]:
import numpy as np
from scipy import stats
th_z = 1.5

for d, s in normal_volume.groupby('device_id'):
    if stats.kstest(s.Volume, stats.norm.cdf).pvalue < 0.05:
        zscore = np.abs((s.Volume - s.Volume.mean()) / s.Volume.std())
        normal_volume.loc[s.index, "zscore"] = zscore > 2
stats_outliers = stats_outliers.merge(normal_volume.groupby('device_id').zscore.sum(), on = 'device_id')

In [ ]:
stats_outliers[(stats_outliers.outlier> 0) | (stats_outliers.zscore)]

In [ ]:
stats_outliers = stats_outliers.groupby(['frequency', 'device_id'], as_index=False)[['outlier', 'zscore']].sum()

In [ ]:
s = stats_outliers.groupby('frequency', as_index=False)[['outlier', 'zscore']].count()
fig = go.Figure()
fig.add_trace(
    go.Bar(name ='IQR_meth', x=s.frequency, y=s.outlier,
        marker={'color': pink}
    )
)
fig.add_trace(
    go.Bar(name="zscore", x=s.frequency, y=s.zscore, marker={'color': blue}
))
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Nombre de PCE ayant des outliers selon la méthode et en fonction de la fréquence",
                  xaxis_title="Fréquence", 
                  yaxis_title="Nombre d'outliers",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
s = stats_outliers.groupby('frequency', as_index=False)[['outlier', 'zscore']].sum()
fig = go.Figure()
fig.add_trace(
    go.Bar(name ='IQR_meth', x=s.frequency, y=s.outlier,
        marker={'color': pink}
    )
)
fig.add_trace(
    go.Bar(name="zscore", x=s.frequency, y=s.zscore, marker={'color': blue}
))
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Somme totale du nombre d'outliers parmi les relevés selon la méthode en fonction de la fréquence",
                  xaxis_title="Fréquence", 
                  yaxis_title="Nombre d'outliers",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
dev = '68ad7b3797243462e20a2abb'
s = normal_volume[normal_volume.device_id==dev].sort_values('payload_timestamp')
anom = s[s.zscore]
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=s.payload_timestamp, y=s.Volume,
        marker={'color': red}
    )
)
fig.add_trace(
    go.Scatter(name="Anomalie", x=anom.payload_timestamp, y=anom.Volume,
                             mode="markers", marker={'color': black, 'size': 4})
)
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Variation sur le temps du volume brut publié pour le PCE {dev}",
                  xaxis_title="Temps", 
                  yaxis_title="Volume brut en m3",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

# <span style="color:purple"> 3 - Recherche d'anomalies parmi les données informatives </span> 

In [ ]:
# Capteurs de consommation
V_reel_info = '64f9dc4e964b1c6b3092aebe'
V_brut_info = '64f9dbc06bd36472e445b9c7'
date_fin_info = 'INFORM_CONSUMPTION_END_DATE'
date_debut_info = 'INFORM_CONSUMPTION_START_DATE'

coef_PTA_info = '64f9dd1fe1aa566bd5c539c8'
coef_PCS_info = '64f9dd576bd364b24d45b9c8'
coef_info = '64f9dd87e1aa563305c539c9'

raison = '64f9db0d2ade76c96cb1e41c'

In [ ]:
dev_freq = client.query_df(f"""
        SELECT DISTINCT device_id, grdf_frequency as frequency
        FROM {grdf_all}
        settings max_rows_to_read = {maxi}""")

In [ ]:
v_raw_df = client.query_df(f"""
        SELECT p.device_id, p.payload_metric_id, p.payload_timestamp, p.payload_value
        from {releves} p
        WHERE payload_source = '{source}'
            AND p.payload_metric_id IN {[V_brut_info, raison, date_fin_info, date_debut_info, coef_PTA_info, coef_PCS_info, coef_info]}
            AND p.payload_timestamp > toDateTime('2024-01-01 12:00:00')
        settings max_rows_to_read = {maxi}""")

In [ ]:
volume_data = v_raw_df.pivot_table(index = ['device_id','payload_timestamp'], columns = "payload_metric_id", values="payload_value", aggfunc='first')\
    .reset_index().rename({V_brut_info:'Volume', raison:'Raison', coef_PTA_info : 'coef_PTA', coef_PCS_info: 'coef_PCS', 
                          coef_info:'coef_info'}, axis=1)
#remplacer les valeurs 'null'
volume_data.loc[volume_data.Volume=='null', 'Volume'] = np.nan

In [ ]:
nbPay = volume_data.groupby('device_id').count().INFORM_CONSUMPTION_END_DATE
volume_data = volume_data.merge(dev_freq[['device_id', 'frequency']], on = 'device_id')

## <span style="color:green"> 1- Détermination des données manquantes </span>

### <span style="color:blue"> a)- Parmi tous les capteurs confondus </span>

In [ ]:
cols = ['Raison', 'coef_PTA', 'coef_PCS', 'coef_info', 'Volume', 'INFORM_CONSUMPTION_END_DATE', 'INFORM_CONSUMPTION_START_DATE']
comb_counts = (volume_data[cols].isna().value_counts()/volume_data.shape[0]*100).reset_index(name='count')
bool_data = 1-comb_counts[cols].astype(int)
comb_labels = bool_data.astype(str).agg(''.join, axis=1)

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.7, 0.3],
    shared_yaxes=True,
    horizontal_spacing=0.05,
    subplot_titles=("Donnée manquante par capteur", "Proportion de donnée manquante")
)

fig.add_trace(
    go.Heatmap(
        z=bool_data.values,
        x=bool_data.columns,
        y=comb_labels,
        colorscale="RdYlGn",
        showscale=False,
        zmin=0, zmax=1
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=comb_counts["count"],
        y=comb_labels,
        orientation='h',
        name="count",
        marker={'color': blue}
    ),
    row=1, col=2
)

fig.update_yaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(title_text="Combinaison de capteur", row=1, col=1)
fig.update_xaxes(title_text="Proportion (échelle logarithmique)", type="log", row=1, col=2)
fig.update_layout(title=f"Proportion de valeurs manquantes pour chaque type de capteur",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1200, height=900,
                template="simple_white")

fig.show()

### <span style="color:blue"> a)- Uniquement parmi les données de volume </span>

In [ ]:
data_vol = volume_data[~volume_data.Volume.isna()]

In [ ]:
cols = ['coef_PTA', 'coef_PCS', 'coef_info', 'Volume', 'INFORM_CONSUMPTION_END_DATE', 'INFORM_CONSUMPTION_START_DATE']
comb_counts = (data_vol[cols].isna().value_counts()/data_vol.shape[0]*100).reset_index(name='count')
bool_data = 1-comb_counts[cols].astype(int)
comb_labels = bool_data.astype(str).agg(''.join, axis=1)

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.7, 0.3],
    shared_yaxes=True,
    horizontal_spacing=0.05,
    subplot_titles=("Donnée manquante par capteur", "Proportion de donnée manquante")
)

fig.add_trace(
    go.Heatmap(
        z=bool_data.values,
        x=bool_data.columns,
        y=comb_labels,
        colorscale="RdYlGn",
        showscale=False,
        zmin=0, zmax=1
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=comb_counts["count"],
        y=comb_labels,
        orientation='h',
        name="count",
        marker={'color': blue}
    ),
    row=1, col=2
)

fig.update_yaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(title_text="Combinaison de capteur", row=1, col=1)
fig.update_xaxes(title_text="Proportion (échelle logarithmique)", type='log', row=1, col=2)
fig.update_layout(title=f"Proportion de valeurs manquantes pour chaque type de capteur, parmi les relevés qui intègrent le volume brut informatif",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1200, height=600,
                template="simple_white")

fig.show()

## <span style="color:green"> 2- Détection des anomalies </span>

In [ ]:
def detect_outliers_iqr(data, k=2.5):
    q25, q75 = np.percentile(data, [25, 75])
    iqr = q75 - q25
    if iqr < 1:
        iqr = k
    cutoff = iqr * k
    thres = q75 + cutoff
    return (data > thres, data - thres)

### <span style="color:blue"> a)- Méthode de l'écart interquartile </span>

  1. On fixe le paramètre k.
> Dans notre cas d'usage, k = 1e4

  3. Calcul de l'écart IQR entre le quantile 25% et le quantile 75% sur une historique 1 an+ de données
  4. Si l'écart est inférieur à 1 alors iqr = k
  5. Toute valeur > q75 + IQR*k est considérée aberrante

Pré-processing :
* Suppression des valeurs négatives
* Suppression des valeurs notées 'null'

In [ ]:
nbPay = data_vol.groupby(['device_id', 'frequency'], as_index = False).agg(
    start=("INFORM_CONSUMPTION_START_DATE", "min"),
    end=("INFORM_CONSUMPTION_END_DATE", "max"),
    nbPay=("Volume", "count")).sort_values('nbPay')
data_vol['Volume'] = data_vol.Volume.astype(float)
# Pour supprimer les valeurs négatives aberrantes
data_vol = data_vol[data_vol.Volume > 0.0]
for d, s in data_vol.groupby('device_id'):
    data_vol.loc[s.index, "outlier_k1e4"] = detect_outliers_iqr(s.Volume.values, k=1e4)
stats_outliers = nbPay.merge(data_vol.groupby('device_id')[['outlier', 'outlier_k100', 'outlier_k1e4']].sum(), on = 'device_id')
stats_outliers['outlier_k1e4'] = stats_outliers.outlier_k1e4.astype(int)

In [ ]:
for d, s in data_vol.groupby('device_id'):
    data_vol.loc[s.index, "outlier_k1e4"] = detect_outliers_iqr(s.Volume.values, k=1e4)
stats_outliers = nbPay.merge(data_vol.groupby('device_id')[['outlier', 'outlier_k100', 'outlier_k1e3']].sum(), on = 'device_id')
stats_outliers['outlier_k1e4'] = stats_outliers.outlier_k1e4.astype(int)

In [ ]:
dev_anom = data_vol[(data_vol.outlier_k100 == 1)].device_id.unique()
print('Nombre de PCE ayant des valeurs aberrantes', len(dev_anom))

In [ ]:
dev_client = client.query_df(f"""
        SELECT DISTINCT device_id, iot_entity
        FROM {equipements}
        WHERE device_id IN {list(dev_anom)}
        settings max_rows_to_read = {maxi}""")

In [ ]:
print('Nombre de clients ayant des PCE avec valeur aberrantes', len(dev_client.iot_entity.unique()))

#### <span style="color:purple"> Analyse des faux négatifs (anomalies manquées)</span>

In [ ]:
data_vol[(data_vol.Volume > 9999) & (data_vol.outlier_k100 == 0)].sort_values('Volume').tail(50)

> Est-ce que ce genre de valeur haute (proche des valeurs identifiées comme outliers) doivent aussi être considérées comme outlier ou bien comportement normal ?

In [ ]:
sample = data_vol[(data_vol.Volume > 9999) & (data_vol.outlier_k100 == 0)].sort_values('Volume').device_id.unique()[:20]
for dev in sample:    
    s = data_vol[data_vol.device_id==dev].sort_values('payload_timestamp')
    anom = s[s.outlier_k100]
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(x=s.payload_timestamp, y=s.Volume,
            marker={'color': red}
        )
    )
    fig.add_trace(
        go.Scatter(name="anomalie", x=anom.payload_timestamp, y=anom.Volume,
                                 mode="markers", marker={'color': blue, 'size': 4})
    )
    fig.update_layout(title=f"Variation sur le temps du volume brut publié pour le PCE {dev}",
                      xaxis_title="Temps", 
                      yaxis_title="Volume brut en m3",
                      font_family="Arial", font_color=black,
                    title_font_family="Times New Roman",
                    title_font_color=black,
                    width=1000, height=400,
                    template="simple_white")
    fig.show()

#### <span style="color:purple"> Analyse des faux positif (anomalies détectées à tort car valeurs faibles) </span>

In [ ]:
data_vol[(data_vol.outlier_k100 == 1) &(data_vol.Volume<100)][['device_id', 'Volume', 'coef_PTA', 'coef_PCS', 'coef_info', \
                                      'INFORM_CONSUMPTION_END_DATE', 'INFORM_CONSUMPTION_START_DATE', \
                                      'frequency', 'outlier_k100', 'outlier_k1e4']].sort_values('Volume').head(50)

In [ ]:
s = data_vol[data_vol.device_id=="64ad484684f4d7e38d5f5251"].sort_values('payload_timestamp')
anom1 = s[s.outlier_k1e4]
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=s.payload_timestamp, y=s.Volume,
        marker={'color': red}
    )
)
fig.add_trace(
    go.Scatter(name="anomalie", x=anom1.payload_timestamp, y=anom1.Volume,
                             mode="markers", marker={'color': blue, 'size': 4})
)
fig.update_layout(title=f"Variation sur le temps du volume brut publié pour le PCE {dev}",
                  xaxis_title="Temps", 
                  yaxis_title="Volume brut en m3",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
for dev in data_vol[data_vol.outlier_k100 == 1].sort_values('Volume').device_id.unique()[:20]:    
    s = data_vol[data_vol.device_id==dev].sort_values('payload_timestamp')
    anom = s[s.outlier_k100]
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(x=s.payload_timestamp, y=s.Volume,
            marker={'color': red}
        )
    )
    fig.add_trace(
        go.Scatter(name="anomalie", x=anom.payload_timestamp, y=anom.Volume,
                                 mode="markers", marker={'color': blue, 'size': 4})
    )
    fig.update_layout(title=f"Variation sur le temps du volume brut publié pour le PCE {dev}",
                      xaxis_title="Temps", 
                      yaxis_title="Volume brut en m3",
                      font_family="Arial", font_color=black,
                    title_font_family="Times New Roman",
                    title_font_color=black,
                    width=1000, height=400,
                    template="simple_white")
    fig.show()

### <span style="color:blue"> b)- Méthode de l'écart interquartile avec fenêtre d'observation réduite à un mois </span>

In [ ]:
data_vol['month'] = data_vol.payload_timestamp.dt.month
data_vol['year'] = data_vol.payload_timestamp.dt.year

In [ ]:
data_vol

#### <span style="color:purple"> Avec k = 100 </span>

In [ ]:
data_vol['Volume'] = data_vol.Volume.astype(float)

In [ ]:
for d, s in data_vol.groupby(['device_id', 'year', 'month']):
    outliers = detect_outliers_iqr(s.Volume.values, k=100)
    data_vol.loc[s.index, "outlier_m_k100"] = outliers[0]
    data_vol.loc[s.index, "dev_m_k100"] = outliers[1]

In [ ]:
data_vol[data_vol.outlier_m_k100 >0]

In [ ]:
data_vol[data_vol.outlier_m_k100 >0].sort_values('dev_m_k100').device_id.unique()[:20]

In [ ]:
for dev in data_vol[data_vol.outlier_m_k100 >0].sort_values('dev_m_k100').device_id.unique()[:20]:
    s = data_vol[data_vol.device_id==dev].sort_values('payload_timestamp')
    anom = s[s.outlier_m_k100]
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(x=s.payload_timestamp, y=s.Volume,
            marker={'color': red}
        )
    )
    fig.add_trace(
        go.Scatter(name="anomalie (fenêtre = 1mois)", x=anom.payload_timestamp, y=anom.Volume,
                                 mode="markers", marker={'color': black, 'size': 4})
    )
    fig.update_layout(title=f"Variation sur le temps du volume brut publié pour le PCE {dev}",
                      xaxis_title="Temps", 
                      yaxis_title="Volume brut en m3",
                      font_family="Arial", font_color=black,
                    title_font_family="Times New Roman",
                    title_font_color=black,
                    width=1000, height=400,
                    template="simple_white")
    fig.show()

### <span style="color:blue"> b)- Test avec un z-score (ne fonctionne que si les données sont normales) </span>

In [ ]:
for d, s in normal_volume.groupby('device_id'):
    if stats.kstest(s.Volume, stats.norm.cdf).pvalue < 0.05:
        zscore = (s.Volume - s.Volume.mean()) / s.Volume.std()
        normal_volume.loc[s.index, "zscore"] = zscore > 2
stats_outliers = stats_outliers.merge(normal_volume.groupby('device_id').zscore.sum(), on = 'device_id')

In [ ]:
stats_outliers = stats_outliers.groupby(['frequency', 'device_id'], as_index=False)[['outlier', 'zscore']].sum()

In [ ]:
s = stats_outliers[(stats_outliers.outlier>0) | (stats_outliers.zscore>0)].groupby('frequency', as_index=False)[['outlier', 'zscore']].count()
fig = go.Figure()
fig.add_trace(
    go.Bar(name ='IQR_meth', x=s.frequency, y=s.outlier,
        marker={'color': pink}
    )
)
fig.add_trace(
    go.Bar(name="zscore", x=s.frequency, y=s.zscore, marker={'color': blue}
))
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Nombre de PCE ayant des outliers selon la méthode et en fonction de la fréquence",
                  xaxis_title="Fréquence", 
                  yaxis_title="Nombre d'outliers",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
s = stats_outliers.groupby('frequency', as_index=False)[['outlier', 'zscore']].sum()
fig = go.Figure()
fig.add_trace(
    go.Bar(name ='IQR_meth', x=s.frequency, y=s.outlier,
        marker={'color': pink}
    )
)
fig.add_trace(
    go.Bar(name="zscore", x=s.frequency, y=s.zscore, marker={'color': blue}
))
# fig.update_yaxes(type="log")
fig.update_layout(title=f"Somme totale du nombre d'outliers parmi les relevés selon la méthode en fonction de la fréquence",
                  xaxis_title="Fréquence", 
                  yaxis_title="Nombre d'outliers",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

In [ ]:
stats_outliers = stats_outliers.merge(normal_volume.groupby('device_id', as_index=False).Volume.count(), on = 'device_id')
stats_outliers.rename({'Volume':'nb_payloads'}, axis=1, inplace=True)

In [ ]:
stats_outliers = stats_outliers.merge(normal_volume.groupby('device_id', as_index=False).Volume.max(), on = 'device_id')

In [ ]:
stats_outliers.sort_values('Volume')

In [ ]:
volume_data[volume_data.CONSUMPTION_START_DATE.isna()].sort_values('payload_timestamp').tail(20)

In [ ]:
s[~s['Volume'].isna()].Volume.astype(int).sort_values()